In [ ]:
from llm_utilis import *

In [ ]:
# load raw data
org_df_raw = pd.read_csv('20251012_bulk_export/organizations.csv')
org_des_df_raw = pd.read_csv('20251012_bulk_export/organization_descriptions.csv')
fund_rounds_df_raw = pd.read_csv('20251012_bulk_export/funding_rounds.csv')

In [ ]:
org_df = org_df_raw[org_df_raw['country_code'].isin(['USA', 'CAN'])]  # filter to US and Canada only
org_df = org_df[org_df['roles'].isin(['company', 'company,investor'])]  # filter to keep only organizations that are not investors
# fileter on founded_on >=2005-01-01
org_df['founded_on'] = pd.to_datetime(org_df['founded_on'], errors='coerce')
org_df = org_df[org_df['founded_on'] >= '2005-01-01']
# exclude companies with  status 'closed' and close_on before 2019-01-01
org_df['close_on'] = pd.to_datetime(org_df['closed_on'], errors='coerce')
org_df = org_df[~((org_df['status'] == 'closed') & (org_df['close_on'] < '2019-01-01'))]

# add description info 
org_df = pd.merge(org_des_df_raw[['uuid','description']], org_df, on='uuid',how='right')


In [ ]:
client = genai.Client(api_key=API_KEY)
creds = service_account.Credentials.from_service_account_file("gemini-key.json")
bucket_name = "ra-crunchbase"

# --- Execution ---

# 1. Estimate Tokens and Create Batches
avg_tokens = estimate_avg_tokens(org_df.sample(100), client)
print(f"Estimated average tokens per prompt: {avg_tokens}")

# Setting max_tokens a bit lower than the model's max for safety (e.g., 4000/1M tokens)
# A JSONL file has a max size of 2GB, but batch size is limited by model memory/token limits.
batches = split_into_token_safe_batches(org_df, avg_tokens, max_tokens=4000) 
print(f"Total batches created: {len(batches)}")

# 2. Write JSONL files
files = write_jsonl_batches(batches, prefix="batch_part")




In [ ]:
# 3. Upload + Run Batch API
# NOTE: Ensure you have enough API quota and the client is authenticated!
# each time only run 30 files to avoid overloading the API
dfs = []
failed_files_overall = []
size = 30
start = 61050
for i in range(start, len(files), size):
    batch_files = files[i:i+size]
    jobs, failed_files = upload_and_run_batch_jobs(batch_files, client) 
    failed_files_overall.extend(failed_files)
    final_results = poll_batch_jobs(jobs, client)
    for k, v in final_results.items():
        file = v['output_file']
        df = extract_output_file_content(client, file)
        dfs.append(df)
    print(f"Batch {i//size + 1}: Completed processing.")
final_df = pd.concat(dfs, ignore_index=True)


In [ ]:
final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv('crunchbase_companies_classificaiton_16.csv', index=False)